# Supervisor agent (Nemotron ReAct loop)

Runs one county-season: scout → route → features → classify → aggregate → critic → (one rerun) → finish.
Tools are **stubs** for now (every result says `"stub": true`). Swap in the real toolbox when P1-P3 and P5 deliver.

Needs `NVIDIA_API_KEY=...` in `.env` at the repo root for cells 3-6. Cells 1-2 cost nothing.

In [ ]:
# 1. Setup: make the agents package importable and work from the workspace root
import os, sys, json
from pathlib import Path
from fieldvision_agents.settings import REPO as ROOT
os.chdir(ROOT)

print('working in', Path.cwd())

In [ ]:
# 2. Offline guardrail tests (scripted fake LLM, no API calls, no credits)
from fieldvision_agents.react import test_supervisor as t
for name, fn in vars(t).items():
    if name.startswith('test_'):
        fn(); print('PASS', name)

In [ ]:
# 3. Which Nemotron models can our key use?
from fieldvision_agents.react.supervisor import NemotronClient
llm = NemotronClient()
print('\n'.join(sorted(m.id for m in llm.client.models.list() if 'nemotron' in m.id.lower())))

In [ ]:
# 4. Real Nemotron, stub tools: critic warns once, then accepts after the rerun
from fieldvision_agents.react.supervisor import run_county
from fieldvision_agents.react.tools_stub import StubToolbox
result = run_county('24021', 2022, StubToolbox(critic_script=('warn', 'accept')), llm)
print(json.dumps(result, indent=2))

In [ ]:
# 5. Real Nemotron, REAL tools + REAL critic (Frederick 2022)
import os, sys, json

from fieldvision.toolbox import Toolbox
from critic_toolbox import CriticToolbox
from fieldvision_agents.react.supervisor import run_county, NemotronClient

tools = CriticToolbox(Toolbox())
result = run_county('24021', 2022, tools, NemotronClient())
print(json.dumps(result, indent=2, default=str)[:3000])

In [ ]:
import json
from fieldvision_agents.react.supervisor import NemotronClient

probe = NemotronClient(use_cache=False)
models = sorted(m.id for m in probe.client.models.list() if "nemotron" in m.id.lower())
tool = [{"type": "function", "function": {"name": "ping", "description": "Reply by calling this.",
         "parameters": {"type": "object", "properties": {"ok": {"type": "boolean"}}, "required": ["ok"]}}}]

for m in models:
    try:
        r = probe.client.chat.completions.create(
            model=m, messages=[{"role": "user", "content": "Call the ping tool with ok=true."}],
            tools=tool, max_tokens=60, temperature=0, timeout=40)
        calls = r.choices[0].message.tool_calls
        print(f"OK     tools={'YES' if calls else 'no '}  {m}")
    except Exception as e:
        msg = str(e)
        status = "NO ACCESS" if "404" in msg or "Not found" in msg else ("NO TOOLS" if "tool" in msg.lower() else "ERROR")
        print(f"{status:9}               {m}")

In [ ]:
# 5. Read the audit log as a table: what the agent did and why
import pandas as pd
log = pd.read_json(result['audit_log'], lines=True)
cols = [c for c in ['step', 'kind', 'tool', 'reason', 'args', 'result'] if c in log]
pd.set_option('display.max_colwidth', 120)
log[cols]

In [ ]:
# 6. Stress test with real Nemotron: critic fails twice, so the agent must abstain
hard = run_county('24021', 2022, StubToolbox(critic_script=('fail', 'fail')), llm)
print(hard['status'], hard['policies'], hard['critic'])